# SpendCoder: fine-tune a transformer with Hugging Face, step by step

**Goal:** a company's accounting team gets thousands of purchase and invoice lines such as *"ANNUAL MAINTENANCE OF HV
SWITCHGEAR"*. Each one must get the right spend category before it is posted or analysed. You will teach a
pretrained language model to do that, test it honestly, and package it for production.

**What you will do (about 30 minutes, most of it waiting):**
1. Get real data: US Department of Energy purchase descriptions with official spend codes
2. Build a simple baseline (TF-IDF + logistic regression): the model to beat
3. See how a tokenizer turns text into numbers
4. Load a pretrained model (DistilRoBERTa) and add a classification head
5. Set the training arguments and understand each one
6. Fine-tune it
7. Evaluate on the newest year it never saw, and compare with the baseline
8. Turn the scores into a business rule: code automatically when confident, otherwise ask a person
9. Try it on your own lines
10. Export to ONNX, check it gives the same answers, and publish to the Hugging Face Hub

**Before you start:** *Runtime → Change runtime type → T4 GPU* (free). Then run the cells from top to bottom.

In [ ]:
!pip -q install datasets onnxscript onnxruntime   # transformers, torch, tokenizers are already in Colab
%cd /content
!test -d SpendCoder && git -C SpendCoder pull -q || git clone -q https://github.com/saeedm31/SpendCoder.git
%cd SpendCoder

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (training will be slow; switch the runtime to T4 GPU)")

## Step 1. Real data

**What:** contracts of the US Department of Energy from [USAspending.gov](https://www.usaspending.gov)
(public domain). Each has a short description written by a buyer, and an official *Product and Service Code*
(PSC). We group the thousands of PSC codes into **14 spend categories**, like a company's spend or GL category
list (see `spendcoder/taxonomy.py`).

**Why this split:** like in production, we train on the past and test on the future. Train = contracts that
started before October 2023, validation = the next 12 months, test = from October 2024. Test lines whose exact
text appears in training are removed, so the test measures new wording, not memory.

This downloads up to 64,000 contracts (around 10 minutes); after removing lines without a code and repeated
texts, about 20,000 labelled lines remain. For a quicker first try, set `PER_YEAR = 3000`.

In [ ]:
from spendcoder import data, metrics
from spendcoder.taxonomy import CATEGORIES, ID2LABEL, LABEL2ID

PER_YEAR = 8000
raw = data.fetch(first_fy=2018, last_fy=2025, max_rows_per_year=PER_YEAR)
df = data.build_dataset(raw)
df.to_csv("data/spend_lines.csv.gz", index=False)
train_df, val_df, test_df = data.split(df)
print(f"train {len(train_df):,} · validation {len(val_df):,} · test {len(test_df):,}")
data.summary(train_df, val_df, test_df)

In [ ]:
# What the model sees (text) and what it must answer (category)
df.sample(10, random_state=1)[["text", "psc", "category"]]

## Step 2. The baseline to beat

**What:** TF-IDF turns each line into word and character counts, weighted by how rare they are; logistic
regression learns which ones point to which category. It trains in seconds on a laptop.

**Why first:** fine-tuning a transformer costs GPU time and is harder to run. It is only worth it if it clearly
beats this. Always measure against a simple baseline.

**The scores:** *accuracy* = share of lines right. *Macro F1* = the average F1 over the 14 categories, so small
categories count as much as big ones (an accounting team cares about the rare ones too).

In [ ]:
from spendcoder.baseline import TfidfBaseline

baseline = TfidfBaseline().fit(train_df["text"], train_df["label"])
baseline_test = metrics.scores(test_df["label"], baseline.predict_proba(test_df["text"]))
baseline_test

## Step 3. Tokenizer: from text to numbers

**What:** a transformer cannot read letters. The tokenizer splits text into *tokens* (whole words or pieces of
words) and maps each to an id in the model's vocabulary. It also adds special tokens: `<s>` at the start and
`</s>` at the end. Rare words like "SWITCHGEAR" are split into known pieces, so nothing is out of vocabulary.

The tokenizer must be the one the model was pretrained with: same vocabulary, same ids.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilroberta-base"   # 82M parameters: a smaller, faster RoBERTa, good for short texts
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = "ANNUAL MAINTENANCE OF HV SWITCHGEAR"
enc = tokenizer(example)
print(tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print(enc["input_ids"])
print(enc["attention_mask"], "<- 1 = real token, 0 = padding")

In [ ]:
# How long are our lines in tokens? This picks MAX_LENGTH (longer lines are cut).
import numpy as np
lengths = [len(tokenizer(t)["input_ids"]) for t in train_df["text"].sample(2000, random_state=0)]
print("95% of lines have at most", int(np.percentile(lengths, 95)), "tokens; the longest:", max(lengths))
MAX_LENGTH = 96   # covers about 95% of lines; the rest are cut

## Step 4. Pretrained model + a new classification head

**What:** `AutoModelForSequenceClassification` loads the pretrained DistilRoBERTa (which already learned English
from a lot of text) and adds a small **classification head**: a layer with one output per category. The body is
pretrained; the head starts random. You will see a warning that some weights are "newly initialized": that is
the head, and it is expected.

**Fine-tuning** = training body and head together on our labelled lines, with a small learning rate, so the
model keeps its language knowledge and adapts it to spend categories. That is why it needs far less data than
training from scratch.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CATEGORIES), id2label=ID2LABEL, label2id=LABEL2ID)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")
print(model.classifier)   # the new head: dense layer -> 14 outputs

## Step 5. Datasets and training arguments

**Datasets:** we convert the tables to Hugging Face `Dataset` objects and tokenize them once. Padding is done per
batch (`DataCollatorWithPadding`): each batch is padded to its longest line, which is faster than padding
everything to 96.

**Training arguments, one by one:**
- `learning_rate=5e-5`: small steps, so the pretrained knowledge is adjusted, not destroyed
- `num_train_epochs=8`: up to eight passes over the training data; `EarlyStoppingCallback(patience=2)` stops
  sooner if validation macro F1 does not improve for two epochs. (A first run with 4 epochs was still improving
  when it stopped: under-trained.)
- `per_device_train_batch_size=32`: lines per training step
- `warmup_steps`: start with an even smaller learning rate for the first 6% of steps, for stable training
- `weight_decay=0.01`: a little regularisation against overfitting
- `eval_strategy="epoch"` + `load_best_model_at_end=True` + `metric_for_best_model="macro_f1"`: score the
  validation year after each epoch and keep the best epoch, not the last
- `fp16=True` on a GPU: half-precision numbers, about twice as fast with the same quality

**Class-weighted loss:** some categories have 20 times more lines than others. With a plain loss the model can
mostly ignore the rare ones, which hurts macro F1. `WeightedTrainer` overrides `compute_loss` so a mistake on a
rare category costs more (square root of the balanced weights, for stable training). The baseline uses balanced
class weights too, so the comparison is fair.

In [ ]:
import math
from datasets import Dataset
from transformers import (DataCollatorWithPadding, EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)
from spendcoder.finetune import class_weights, compute_metrics

set_seed(42)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize, batched=True)

class WeightedTrainer(Trainer):
    """Trainer with class-weighted cross-entropy: mistakes on rare categories cost more."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits.float(), labels, weight=self.class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

weights = class_weights(train_df["label"])
print("loss weight per category:", dict(zip(CATEGORIES, weights.round(2))))

EPOCHS, BATCH = 8, 32
steps = math.ceil(len(train_ds) / BATCH) * EPOCHS
args = TrainingArguments(
    output_dir="models/spendcoder",
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=5e-5, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2,
    warmup_steps=int(0.06 * steps), weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    save_total_limit=1, fp16=torch.cuda.is_available(),
    logging_steps=50, report_to="none", seed=42,
)
trainer = WeightedTrainer(
    class_weights=weights,
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,          # accuracy, macro F1, weighted F1 on validation
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

## Step 6. Fine-tune

**What happens in each step:** a batch of 32 lines goes through the model, the model outputs a score per category,
the *loss* (cross-entropy) measures how wrong it was, back-propagation computes how each weight should change, and
the optimizer (AdamW) moves the weights a little. The training loss should go down; after each epoch you see the
validation scores. About 10 to 15 minutes on a T4.

In [ ]:
trainer.train()
trainer.save_model("models/spendcoder")
tokenizer.save_pretrained("models/spendcoder")

## Step 7. Test on the newest year, compare with the baseline

The test year was never used for training or for choosing the best epoch, so these are the honest numbers.

In [ ]:
import pandas as pd
from spendcoder.finetune import predict_proba

model = trainer.model
val_p = predict_proba(model, tokenizer, val_df["text"].tolist(), max_length=MAX_LENGTH)
test_p = predict_proba(model, tokenizer, test_df["text"].tolist(), max_length=MAX_LENGTH)
finetuned_test = metrics.scores(test_df["label"], test_p)
pd.DataFrame({"TF-IDF baseline": baseline_test, "Fine-tuned DistilRoBERTa": finetuned_test}).round(3)

In [ ]:
# Per category: where is it strong, where is it weak?
rep = pd.DataFrame(metrics.report(test_df["label"], test_p)).T
rep.loc[CATEGORIES, ["precision", "recall", "f1-score", "support"]].round(2)

In [ ]:
# Error analysis: look at real mistakes. Often the label is debatable, or the text is too vague to decide.
test_view = test_df.assign(predicted=[CATEGORIES[i] for i in test_p.argmax(1)], confidence=test_p.max(1))
wrong = test_view[test_view["predicted"] != test_view["category"]]
wrong.sort_values("confidence", ascending=False)[["text", "category", "predicted", "confidence"]].head(15)

## Step 8. From scores to a business rule

No model is right 100% of the time, so in production it should not post everything blindly. The rule:
**code a line automatically if the model's confidence is above a threshold; otherwise send it to a person with the
top 3 suggestions.** We pick the threshold on the *validation* year so that automatic codings reach 95%
accuracy, then check it on the test year.

In [ ]:
threshold = metrics.threshold_for_accuracy(val_df["label"], val_p, target=0.95)
print("Review threshold:", threshold)
pd.DataFrame(metrics.automation(test_df["label"], test_p, thresholds=sorted({0.5, 0.7, 0.9, threshold})))

## Step 9. Try your own lines

In [ ]:
from spendcoder.predictor import decide

mine = ["Annual maintenance contract for production line pumps",
        "Dell Latitude laptops and docking stations",
        "Asbestos removal and soil remediation at warehouse site",
        "Consulting support for SAP S/4HANA finance migration",
        "Office cleaning and security guard services"]
for text, p in zip(mine, predict_proba(model, tokenizer, mine, max_length=MAX_LENGTH)):
    d = decide(text, p, CATEGORIES, threshold)
    print(f"{d['category']:<40} {d['confidence']:.2f} {'REVIEW' if d['needs_review'] else 'auto  '}  {text}")

## Step 10. Package for production: ONNX + Hugging Face Hub

**ONNX:** a standard model format. Exported once, it runs with `onnxruntime` and the `tokenizers` library only:
no PyTorch, small Docker image, fast on CPU. We check that ONNX and PyTorch give the same probabilities.

**Hugging Face Hub:** a versioned home for the model (like GitHub for models), with a model card that says what it
does, how it was tested and where not to use it. The API and the demo download it from there.

To publish: create a free account at huggingface.co, then *Settings → Access Tokens → New token (write)*. When the
login box appears below, paste the token there (never in code or chat).

In [ ]:
import json, os
from spendcoder import export
from spendcoder.modelcard import model_card

onnx_dir = "models/spendcoder/onnx"
export.export_onnx(model, tokenizer, onnx_dir, threshold=threshold, max_length=MAX_LENGTH,
                   metrics={"baseline_test": baseline_test, "finetuned_test": finetuned_test})
print("ONNX vs PyTorch, largest probability difference:",
      export.check_parity(model, tokenizer, onnx_dir, test_df["text"].head(200).tolist()))

results = {"rows": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
           "baseline_test": baseline_test, "finetuned_test": finetuned_test, "threshold": threshold,
           "finetuned_val": metrics.scores(val_df["label"], val_p),
           "automation_test": metrics.automation(test_df["label"], test_p, thresholds=sorted({0.5, 0.7, 0.9, threshold})),
           "per_category_test": metrics.report(test_df["label"], test_p)}
os.makedirs("reports", exist_ok=True)
json.dump(results, open("reports/finetuned.json", "w"), indent=2)
open("models/spendcoder/README.md", "w").write(model_card(results))
print(open("models/spendcoder/README.md").read()[:1500])

In [ ]:
# Production-style inference: ONNX on CPU, no PyTorch. How fast is it?
import time
from spendcoder.predictor import Predictor

pred = Predictor.load(onnx_dir)
lines = test_df["text"].head(256).tolist()
start = time.time()
out = pred.classify(lines)
print(f"{len(lines)} lines in {time.time() - start:.2f} s on CPU")
out[0]

In [ ]:
# Publish to the Hugging Face Hub (paste your write token into the box that appears)
from huggingface_hub import HfApi, notebook_login
notebook_login()

In [ ]:
api = HfApi()
REPO_ID = f"{api.whoami()['name']}/spendcoder-distilroberta"   # your username, from the login above
api.create_repo(REPO_ID, exist_ok=True)
model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)
api.upload_folder(repo_id=REPO_ID, folder_path=onnx_dir, path_in_repo="onnx")
api.upload_file(repo_id=REPO_ID, path_or_fileobj="models/spendcoder/README.md", path_in_repo="README.md")
print(f"https://huggingface.co/{REPO_ID}")

In [ ]:
# Keep the results: download reports/finetuned.json and send it back (or commit it to the repo)
from google.colab import files
files.download("reports/finetuned.json")

## What you just did

- Fine-tuned a pretrained transformer (DistilRoBERTa) with Hugging Face **Transformers** (`Trainer` API) for
  14-class text classification of real purchase lines
- Built a **baseline** first and compared on a **time-based** hold-out (newest year, no duplicate texts)
- Chose the best epoch on validation with **early stopping**, used **fp16** on a GPU
- Handled **class imbalance** with a class-weighted loss (custom `compute_loss` in a `Trainer` subclass)
- Looked at **per-category** scores and **real errors**
- Turned confidence into a **human-in-the-loop** rule with a threshold chosen on validation
- **Exported to ONNX**, checked parity, and published the model with a **model card** on the Hugging Face Hub;
  the repo serves it with **FastAPI in Docker**